In [162]:
import pandas as pd 
import numpy as np 
from sklearn import linear_model

df = pd.read_csv('C:/Users/Robert/projects/notes/data/healthcare-dataset-stroke-data.csv')
# 5110 rows excl colnames => m
# 12 cols excl id => n
m, n = df.shape
print(f"{m} rows, {n} cols")

df = df.dropna(axis=0) # drops missing bmi, 4909 rows left



def one_hot(df: pd.DataFrame, col: str):
    Y = df[col]
    pos = df.columns.get_loc(col)
    print(pos)
    labels = df[col].unique()

    for label in labels:
        df.insert(pos, f"{col}_{label}", (Y == label).astype(int))  # converts nx1 boolean mask (Y==label) into 1 / 0
    df.drop(columns=col, inplace=True)
    return None


one_hot(df, 'smoking_status')

df.drop(columns='work_type', inplace=True)

df["gender"] = np.where(df['gender'] == 'Male', 1, 0) # if true set to 1, else 0
df["ever_married"] = np.where(df['ever_married'] == 'Yes', 1, 0)
df["Residence_type"] = np.where(df['Residence_type'] == 'Urban', 1, 0)

df

5110 rows, 12 cols
10


,id,gender,age,hypertension,heart_disease,ever_married,Residence_type,avg_glucose_level,bmi,smoking_status_Unknown,smoking_status_smokes,smoking_status_never smoked,smoking_status_formerly smoked,stroke
0,9046,1,67.0,0,1,1,1,228.69,36.6,0,0,0,1,1
2,31112,1,80.0,0,1,1,0,105.92,32.5,0,0,1,0,1
3,60182,0,49.0,0,0,1,1,171.23,34.4,0,1,0,0,1
4,1665,0,79.0,1,0,1,0,174.12,24.0,0,0,1,0,1
5,56669,1,81.0,0,0,1,1,186.21,29.0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5104,14180,0,13.0,0,0,0,0,103.08,18.6,1,0,0,0,0
5106,44873,0,81.0,0,0,1,1,125.20,40.0,0,0,1,0,0
5107,19723,0,35.0,0,0,1,0,82.99,30.6,0,0,1,0,0
5108,37544,1,51.0,0,0,1,0,166.29,25.6,0,0,0,1,0


In [163]:
X = df.iloc[:, 1:13].to_numpy().T
Y = df[['stroke']].to_numpy().T  # logit in sklearn needs 2d array for x but a 1d for y FOR SOME REASON ?

CUTOFF = 3500
X_train = X[:, 0:CUTOFF]
X_test = X[:, CUTOFF:m]
Y_train = Y[:, 0:CUTOFF]
Y_test = Y[:, CUTOFF:m]

print(X.shape)
print(Y.shape)



(12, 4909)
(1, 4909)


In [164]:
layer_dims = [X_train.shape[0], 10, 20, Y_train.shape[0]]  # first layer same dims as input, last layer same dims as output


parameters = {}

def init_parameters(layer_dims: list):

    L = len(layer_dims) - 1             # vvv
    
    for l in range(1, L + 1):           # |||-> ensures we don't init params for the 0th layer or the last layer (they don't need them)

        parameters["W" + str(l)] = np.random.randn(layer_dims[l], layer_dims[l-1]) / np.sqrt(layer_dims[l-1])

        # to be multipliable, weights are shaped (n of neurons in this layer) x (n of neurons previous layer) 
        #  |-> division by sqrt is to deal with gradient vanishing problem. standrd workaround
        parameters["b" + str(l)] = np.zeros((layer_dims[l], 1))  # biases are column vecs

    return parameters

# e.g. init_parameter([2, 4, 10, 4]) will generate W1, b1, W2, b2, W3, b3 to cover the "gaps" between these 4 layers


In [165]:
def relu(Z):
    A = np.maximum(0,Z)
    return A

def sigmoid(Z):
    A = 1/(1+np.exp(-Z))
    return A

def softmax(z):
    expZ = np.exp(z)
    return expZ/(np.sum(expZ, 0))

def tanh(x):
    return np.tanh(x)

def derivative_relu(Z):
    return np.array(Z > 0, dtype = 'float')

def derivative_tanh(x):
    return (1 - np.power(x, 2))

In [166]:
def forward_prop(X, parameters, activation = 'relu'):
    L = len(parameters) // 2    # just a way to get L from inside the func
    forward_cache = {}
    forward_cache['A0'] = X     # take input layer

    for l in range(1, L):                                                                # vvv prev layer
        forward_cache['Z' + str(l)] = parameters['W' + str(l)].dot(forward_cache['A' + str(l-1)]) + parameters['b' + str(l)]
        if activation == 'relu':
            forward_cache['A' + str(l)] = relu(forward_cache['Z' + str(l)])
        else:
            forward_cache['A' + str(l)] = tanh(forward_cache['Z' + str(l)])


    forward_cache['Z' + str(L)] = parameters['W' + str(L)].dot(forward_cache['A' + str(L-1)]) + parameters['b' + str(L)]
    # for output layer use a different function
    if forward_cache['Z' + str(L)].shape[0] == 1:           # maybe check last layer dim for 1 instead.
        
        forward_cache['A' + str(L)] = sigmoid(forward_cache['Z' + str(L)])
    else:
        forward_cache['A' + str(L)] = softmax(forward_cache['Z' + str(L)])
        
    return forward_cache['A' + str(L)], forward_cache



def compute_cost(AL, Y):
    m = Y.shape[1]

    if Y.shape[1] == 1:
        cost = (1./m) * (-np.dot(Y, np.log(AL).T) - np.dot(1-Y, np.log(1-AL).T))
    else:
        cost = -(1./m) * np.sum(Y * np.log(AL))

    cost = np.squeeze(cost)

    return cost

In [167]:
def backward_prop(AL, Y, parameters, forward_cache, activation='relu'):
    grads = {}
    L = len(parameters) // 2
    m = AL.shape[1]

    grads['dZ' + str(L)] = AL - Y
    grads['dW' + str(L)] = 1./m * np.dot(grads['dZ' + str(L)], forward_cache['A' + str(L-1)].T)  # 1./3 = 0.333333, 1/3 = 0 
    grads['db' + str(L)] = 1./m * np.sum(grads['dZ' + str(L)], axis=1, keepdims=True)

    for l in reversed(range(1, L)):
        if activation == 'relu':
            grads['dZ' + str(l)] = np.dot(parameters['W' + str(l+1)].T, grads['dZ' + str(l+1)]) * derivative_relu(forward_cache['A' + str(l)])
        else:
            grads['dZ' + str(l)] = np.dot(parameters['W' + str(l+1)].T, grads['dZ' + str(l+1)]) * derivative_tanh(forward_cache['A' + str(l)])

        grads['dW' + str(l)] = 1./m * np.dot(grads['dZ' + str(l)], forward_cache['A' + str(l-1)].T)  # 1./3 = 0.333333, 1/3 = 0 
        grads['db' + str(l)] = 1./m * np.sum(grads['dZ' + str(l)], axis=1, keepdims=True)

    return grads


In [168]:
def update_params(parameters, grads, learning_rate):
    L = len(parameters) // 2
    
    for l in range(1,L+1):          # mind the +1 !!! also use l+1 for indexing since we are finding the NEXT layer's params using our initialized params

        parameters['W' + str(l)] = parameters['W' + str(l)] - learning_rate*grads['dW' + str(l)]
        parameters['b' + str(l)] = parameters['b' + str(l)] - learning_rate*grads['db' + str(l)]

    return parameters


In [169]:
def predict(X, y, parameters, activation):

    m = X.shape[1]

    y_pred, caches = forward_prop(X, parameters, activation)
    
    if y.shape[0] == 1:
        y_pred = np.array(y_pred > 0.5, dtype = 'float')  # if > 0.5 then True which will convert to 1 via float dtype
    else:
        y = np.argmax(y, 0) # with multiclass, pick the index of the largest predicted prob (as it was one-hot encoded)
        y_pred = np.argmax(y_pred, 0)   
    
    return np.round(np.sum((y_pred == y)/m), 2)

In [170]:
def neural_net(X, Y, layer_dims, learning_rate, activation='relu', num_iterations = 200):

    parameters = init_parameters(layer_dims)

    for i in range(0, num_iterations):

        AL, forward_cache = forward_prop(X, parameters, activation)
        
        cost = compute_cost(AL, Y)
        grads = backward_prop(AL, Y, parameters, forward_cache, activation)

        parameters = update_params(parameters, grads, learning_rate)

        if i % (num_iterations/10) == 0:
            print(f"iter: {i}, cost: {cost}, train_acc: {predict(X_train, Y_train, parameters, activation)}, test_acc: {predict(X_test, Y_test, parameters, activation)}")

    return parameters



layer_dims = [X_train.shape[0], 20, 10, Y_train.shape[0]]
lr = 0.0001
iters = 10000

parameters = neural_net(X_train, Y_train, layer_dims, lr, 'relu', iters)



iter: 0, cost: 0.4686595645322571, train_acc: 0.94, test_acc: 1.0
iter: 1000, cost: 0.20887096047227088, train_acc: 0.94, test_acc: 1.0
iter: 2000, cost: 0.19156186707258663, train_acc: 0.94, test_acc: 1.0
iter: 3000, cost: 0.17898013441632915, train_acc: 0.94, test_acc: 1.0
iter: 4000, cost: 0.16978571773118822, train_acc: 0.94, test_acc: 1.0
iter: 5000, cost: 0.16340013901315528, train_acc: 0.94, test_acc: 1.0
iter: 6000, cost: 0.1588702224002787, train_acc: 0.94, test_acc: 1.0
iter: 7000, cost: 0.15564231859602537, train_acc: 0.94, test_acc: 1.0
iter: 8000, cost: 0.15365376518344187, train_acc: 0.94, test_acc: 1.0
iter: 9000, cost: 0.1522958673945485, train_acc: 0.94, test_acc: 1.0
